In [1]:
import pathlib
import sys

import duckdb
import pandas as pd

sys.path.insert(0, str(pathlib.Path('../src').resolve()))
import irp.config as _config

cfg = _config.load()
_ROOT = pathlib.Path(_config.__file__).parents[2]
DB = str((_ROOT / cfg['store']['db_path']).resolve())


def load_tables(*names: str) -> dict[str, pd.DataFrame]:
    with duckdb.connect(DB, read_only=True) as con:
        return {name: con.execute(f'SELECT * FROM "{name}"').df() for name in names}


print('DB:', DB)

DB: /mnt/Dev/active_python_projects/investment_research_platform/data/irp.duckdb


In [2]:
data = load_tables('income', 'balance', 'cashflow', 'companies', 'industries')
for name, df in data.items():
    print(f'{name:12s}: {len(df):,} rows')

income      : 68,495 rows
balance     : 68,486 rows
cashflow    : 68,486 rows
companies   : 6,556 rows
industries  : 74 rows


In [3]:
from irp.quality import run

findings = run(data)
print(f'Total findings: {len(findings):,}')
findings.head()

Total findings: 35,558


,table,ticker,variant,report_date,column,value,rule,detail,severity
0,balance,AGR,A,2023-12-31,"Total Assets, Total Liabilities, Total Equity",0.023497,accounting_identity,"Assets=43989000000, Liab+Eq=42955378674, rel_e...",error
1,balance,AIRC,A,2020-12-31,"Total Assets, Total Liabilities, Total Equity",0.013143,accounting_identity,"Assets=6229278000, Liab+Eq=6147407870, rel_err...",error
2,balance,AIRC,A,2021-12-31,"Total Assets, Total Liabilities, Total Equity",0.031869,accounting_identity,"Assets=6440360000, Liab+Eq=6235109012, rel_err...",error
3,balance,AIRC,A,2022-12-31,"Total Assets, Total Liabilities, Total Equity",0.037331,accounting_identity,"Assets=6551883000, Liab+Eq=6307295128, rel_err...",error
4,balance,AIRC,A,2023-12-31,"Total Assets, Total Liabilities, Total Equity",0.044646,accounting_identity,"Assets=6134752000, Liab+Eq=5860857779, rel_err...",error


In [4]:
# Summary: findings per rule + severity
(
    findings.groupby(['severity', 'rule'])
    .size()
    .rename('count')
    .reset_index()
    .sort_values(['severity', 'count'], ascending=[True, False])
)

,severity,rule,count
1,error,impossible_value,259
0,error,accounting_identity,251
2,warning,sector_outlier,21059
3,warning,sudden_jump,13989


In [5]:
# Drill-down: accounting identity violations
findings[findings['rule'] == 'accounting_identity'].sort_values(
    'value', ascending=False
)

,table,ticker,variant,report_date,column,value,rule,detail,severity
38,balance,OMEX,A,2021-12-31,"Total Assets, Total Liabilities, Total Equity",4.087908,accounting_identity,"Assets=8908887, Liab+Eq=45327595, rel_err=408.79%",error
39,balance,OMEX,A,2022-12-31,"Total Assets, Total Liabilities, Total Equity",3.12382,accounting_identity,"Assets=13870827, Liab+Eq=57200799, rel_err=312...",error
37,balance,OMEX,A,2020-12-31,"Total Assets, Total Liabilities, Total Equity",2.570473,accounting_identity,"Assets=11759464, Liab+Eq=41986854, rel_err=257...",error
40,balance,OMEX,A,2023-12-31,"Total Assets, Total Liabilities, Total Equity",2.351466,accounting_identity,"Assets=22752297, Liab+Eq=76253552, rel_err=235...",error
36,balance,MTTR,A,2020-12-31,"Total Assets, Total Liabilities, Total Equity",2.28828,accounting_identity,"Assets=71852000, Liab+Eq=-92565481, rel_err=22...",error
...,...,...,...,...,...,...,...,...,...
110,balance,ENFN,Q,2021-09-30,"Total Assets, Total Liabilities, Total Equity",0.011269,accounting_identity,"Assets=49631000, Liab+Eq=50190272, rel_err=1.13%",error
71,balance,AAMC,Q,2023-09-30,"Total Assets, Total Liabilities, Total Equity",0.011257,accounting_identity,"Assets=50001000, Liab+Eq=50563862, rel_err=1.13%",error
10,balance,BRST,A,2021-12-31,"Total Assets, Total Liabilities, Total Equity",0.010808,accounting_identity,"Assets=251669000, Liab+Eq=254389128, rel_err=1...",error
55,balance,SQSP,A,2020-12-31,"Total Assets, Total Liabilities, Total Equity",0.010289,accounting_identity,"Assets=306766000, Liab+Eq=309922462, rel_err=1...",error


In [6]:
# Drill-down: impossible values
findings[findings['rule'] == 'impossible_value']

,table,ticker,variant,report_date,column,value,rule,detail,severity
251,income,AAMC,A,2023-12-31,Revenue,-17322891.0,impossible_value,Revenue is negative,error
252,income,ALT,A,2022-12-31,Revenue,-68000.0,impossible_value,Revenue is negative,error
253,income,BENF,A,2022-03-31,Revenue,-66618000.0,impossible_value,Revenue is negative,error
254,income,BENF,A,2023-03-31,Revenue,-104903000.0,impossible_value,Revenue is negative,error
255,income,BENF,A,2024-03-31,Revenue,-98908000.0,impossible_value,Revenue is negative,error
...,...,...,...,...,...,...,...,...,...
505,income,XPO,Q,2021-12-31,Revenue,-2243000000.0,impossible_value,Revenue is negative,error
506,income,Z,Q,2020-12-31,Revenue,-926865000.0,impossible_value,Revenue is negative,error
507,income,ZIMV,Q,2021-12-31,Revenue,-280768000.0,impossible_value,Revenue is negative,error
508,income,ZIMV,Q,2022-12-31,Revenue,-221450000.0,impossible_value,Revenue is negative,error


In [7]:
# Drill-down: sector outliers
findings[findings['rule'] == 'sector_outlier'].sort_values(
    'value', key=abs, ascending=False
)

,table,ticker,variant,report_date,column,value,rule,detail,severity
24903,income,JUPW,Q,2024-12-31,Net Income,-236151.144,sector_outlier,"IQR-dist=-236151.1, sector=Healthcare, value=-...",warning
32698,income,JUPW,Q,2024-12-31,Operating Income (Loss),-204008.443,sector_outlier,"IQR-dist=-204008.4, sector=Healthcare, value=-...",warning
31246,income,CRSP,Q,2021-06-30,Operating Income (Loss),30297.497,sector_outlier,"IQR-dist=30297.5, sector=Healthcare, value=762...",warning
23323,income,CRSP,Q,2021-06-30,Net Income,28508.039,sector_outlier,"IQR-dist=28508.0, sector=Healthcare, value=759...",warning
31248,income,CRSP,Q,2021-12-31,Operating Income (Loss),-20612.093,sector_outlier,"IQR-dist=-20612.1, sector=Healthcare, value=-5...",warning
...,...,...,...,...,...,...,...,...,...
34734,income,TDG,Q,2022-03-31,Operating Income (Loss),3.001,sector_outlier,"IQR-dist=3.0, sector=Industrials, value=520000000",warning
20627,income,GLW,A,2024-12-31,Net Income,3.001,sector_outlier,"IQR-dist=3.0, sector=Technology, value=506000000",warning
24243,income,GEN,Q,2024-09-30,Net Income,3.0,sector_outlier,"IQR-dist=3.0, sector=Technology, value=161000000",warning
31991,income,FSLR,Q,2023-06-30,Operating Income (Loss),3.0,sector_outlier,"IQR-dist=3.0, sector=Technology, value=203970000",warning


In [8]:
# Drill-down: sudden jumps
findings[findings['rule'] == 'sudden_jump'].sort_values(
    'value', key=abs, ascending=False
)

,table,ticker,variant,report_date,column,value,rule,detail,severity
1331,income,JUPW,Q,2024-12-31,Revenue,9669139.664,sudden_jump,Revenue: 110213 → 1065665000000 (966913966.4%),warning
7151,income,HTA,Q,2020-12-31,Net Income,2406497.3333,sudden_jump,Net Income: -30 → 72194890 (240649733.3%),warning
14068,balance,CRSP,Q,2021-03-31,Total Assets,1071085.6613,sudden_jump,Total Assets: 1827966 → 1957910000000 (1071085...,warning
740,income,AVYA,Q,2022-03-31,Revenue,1004206.5736,sudden_jump,Revenue: 713 → 716000000 (100420657.4%),warning
13979,balance,AVYA,Q,2021-12-31,Total Assets,983624.731,sudden_jump,Total Assets: 5985 → 5887000000 (98362473.1%),warning
...,...,...,...,...,...,...,...,...,...
14125,balance,FNVT,A,2024-12-31,Total Assets,-0.8003,sudden_jump,Total Assets: 51237770 → 10233676 (-80.0%),warning
11377,income,SENEA,A,2023-03-31,Net Income,-0.8002,sudden_jump,Net Income: 46200000 → 9231000 (-80.0%),warning
1716,income,RKT,Q,2022-03-31,Revenue,-0.8001,sudden_jump,Revenue: 7647193000 → 1529046000 (-80.0%),warning
2542,income,AMKR,Q,2025-03-31,Net Income,-0.8,sudden_jump,Net Income: 105649000 → 21128000 (-80.0%),warning


In [9]:
# Export all findings for manual review
out = pathlib.Path('../data/flagged_anomalies.csv')
findings.to_csv(out, index=False)
print(f'Exported {len(findings):,} findings → {out.resolve()}')

Exported 35,558 findings → /mnt/Dev/active_python_projects/investment_research_platform/data/flagged_anomalies.csv
